# Symbolic Differentiation 

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want to implement a function `diff` that takes one argument:
  - The argument `expr` represents an *arithmetic expression*.
    Here an arithmetic expression is any string that is build from variable and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form 
    $a \texttt{**} b$ is interpreted as $a^b$.          
    Furthermore, if $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff(expr)` will then compute the derivative of `expr` with respect to the variable `x`.  For example, the function call 
`diff("x * exp(x)")` will compute the output
`1 * exp(x) + x * exp(x)` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

This file shows the implementation of a program that can perform *symbolic differentiation* using `Lark`.
We will manually convert the parsed `lark.Tree` into nested tuples so that our symbolic differentiation function can leverage Python's structural pattern matching.

## Specification of the Grammar and Parser

We use `lark` to define our EBNF grammar. The rules inherently enforce mathematical precedence. We include a `?unary` rule to handle unary minus correctly relative to exponentiation and multiplication.

In [ ]:
from lark import Lark, Token

In [ ]:
calc_grammar = """
?start: expr

?expr: expr "+" product   -> add
     | expr "-" product   -> sub
     | product

?product: product "*" unary -> mul
        | product "/" unary -> div
        | unary

?unary: "-" unary           -> neg
      | factor

?factor: base "**" unary    -> pow
       | base

?base: "exp" "(" expr ")"    -> exp
     | "ln" "(" expr ")"     -> ln
     | "(" expr ")"
     | NUMBER                -> number
     | "x"                   -> var

%import common.NUMBER
%import common.WS
%ignore WS
"""

In [ ]:
parser = Lark(calc_grammar, parser='lalr')

## AST Construction

The function below turns the `lark.Tree` into a nested tuple representation that our `diff` function expects, now supporting the `neg` operator.

In [ ]:
def ast_to_tuples(node):
    if isinstance(node, Token):
        return str(node)
    
    match node.data:
        case 'add':    return ('+', ast_to_tuples(node.children[0]), ast_to_tuples(node.children[1]))
        case 'sub':    return ('-', ast_to_tuples(node.children[0]), ast_to_tuples(node.children[1]))
        case 'mul':    return ('*', ast_to_tuples(node.children[0]), ast_to_tuples(node.children[1]))
        case 'div':    return ('/', ast_to_tuples(node.children[0]), ast_to_tuples(node.children[1]))
        case 'pow':    return ('**', ast_to_tuples(node.children[0]), ast_to_tuples(node.children[1]))
        case 'neg':    return ('neg', ast_to_tuples(node.children[0]))
        case 'exp':    return ('exp', ast_to_tuples(node.children[0]))
        case 'ln':     return ('ln', ast_to_tuples(node.children[0]))
        case 'number': return float(node.children[0]) if '.' in node.children[0] else int(node.children[0])
        case 'var':    return 'x'

def parse(s):
    tree = parser.parse(s)
    return ast_to_tuples(tree)

In [ ]:
parse('ln(x ** x) + exp(x * -x)')

## Symbolic Differentiation

Now we are ready to implement the function `diff` that takes an expression possibly containing the variable `x`.
It computes the derivative of the given expression with respect to `x`.
<ol>
<li>  The lines 4 - 6 implement the rule: 
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  Line 7 - 9 implement the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  Line 10 - 12 deals with the product rule. 
      </li>
<li>  Line 13 - 15 deals with the quotient rule. 
      </li>
<li>  Line 16 - 18 deals with exponentiation trick (rewriting using ln and exp).
      </li>
<li>  Line 19 - 20 deals with unary negation: 
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(-f(x)\bigr) = -\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)$$
      </li>
<li>  Line 21 - 23 deals with natural logarithm.
      </li>
<li>  Line 24 - 26 deals with the exponential function.
      </li>
<li>  Otherwise, the expression is assumed to be a constant and hence we return 0.
      </li>
</ol>


In [ ]:
def diff(e):
    "differentiate the expression e with respect to the variable x"
    match e:
        case ('+', f, g):
            fs, gs = diff(f), diff(g)
            return ('+', fs, gs)
        case ('-', f, g):
            fs, gs = diff(f), diff(g)
            return ('-', fs, gs)
        case ('*', f, g):
            fs, gs = diff(f), diff(g)
            return ('+', ('*', fs, g), ('*', f, gs))
        case ('/', f, g):
            fs, gs = diff(f), diff(g)
            return ('/', ('-', ('*', fs, g), ('*', f, gs)), ('*', g, g))
        case ('**', f, g):
            return diff(('exp', ('*', g, ('ln', f))))
        case ('neg', f):
            return ('neg', diff(f))
        case ('ln', f):
            fs = diff(f) 
            return ('/', fs, f)
        case ('exp', f):
            fs = diff(f)
            return ('*', fs, e)
        case 'x':
            return 1
    return 0

### Summary of Simplification Rules

The `simplify` function applies the following algebraic identities to transform expression trees into their canonical, reduced forms. Simplification is performed bottom-up, with recursive re-evaluation to ensure chained simplifications (e.g., `--a -> a`) are fully applied.

#### 1. Constant Folding
* All arithmetic operations (`+`, `-`, `*`, `/`, `**`) are evaluated whenever both operands are numeric constants.

#### 2. Unary Negation Folding
* $-(-a) \rightarrow a$
* $-0 \rightarrow 0$
* $-a$ (where $a$ is a constant) $\rightarrow$ numeric negation

#### 3. Addition & Subtraction Identities
* $a + 0 = a$
* $0 - a = -a$
* $a - 0 = a$
* $a - a = 0$
* $a + (-b) \rightarrow a - b$
* $-a + b \rightarrow b - a$
* $a - (-b) \rightarrow a + b$

#### 4. Multiplication & Reciprocal Rules
* $a \cdot 0 = 0$
* $a \cdot 1 = a$
* $-1 \cdot a \rightarrow -a$
* $a \cdot \frac{1}{a} = 1$
* $-a \cdot \frac{1}{a} = -1$
* $a \cdot (-\frac{1}{a}) = -1$

#### 5. Power Derivative Reduction
*Used to simplify chain rule outputs:*
* $a \cdot \frac{1}{f} \cdot f^b \rightarrow a \cdot f^{b-1}$
* $\frac{1}{f} \cdot f^b \rightarrow f^{b-1}$
* *(Special Case $b=2$)*: $a \cdot \frac{1}{f} \cdot f^2 \rightarrow a \cdot f$

#### 6. Exponentiation, Logarithmic & Exponential Identities
* $a^0 = 1$, $a^1 = a$, $0^a = 0$, $1^a = 1$
* $\ln(1) = 0$
* $\ln(\exp(a)) = a$
* $\exp(0) = 1$
* $\exp(\ln(a)) = a$
* $\exp(a \cdot \ln(f)) \rightarrow f^a$

In [ ]:
def simplify(e):
    """Recursively simplifies an abstract syntax tree representing an arithmetic expression."""
    if isinstance(e, (int, float, str)):
        return e

    # 1. Recursively simplify sub-expressions first (bottom-up)
    match e:
        case (op, left, right):
            s_left = simplify(left)
            s_right = simplify(right)
            e_sim = (op, s_left, s_right)
        case (op, arg):
            s_arg = simplify(arg)
            e_sim = (op, s_arg)
        case _:
            return e

    # 2. Apply algebraic simplification rules
    match e_sim:
        # Unary Negation Folding
        case ('neg', a) if isinstance(a, (int, float)): return -a
        case ('neg', ('neg', a)): return a
        case ('neg', 0): return 0

        # Constant Folding
        case (op, a, b) if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            match op:
                case '+': return a + b
                case '-': return a - b
                case '*': return a * b
                case '/' if b != 0: return a / b
                case '**': return a ** b
                case _: return e_sim

        # Addition & Subtraction
        case ('+', 0, a) | ('+', a, 0): return a
        case ('+', a, ('neg', b)): return simplify(('-', a, b))
        case ('+', ('neg', a), b): return simplify(('-', b, a))
        
        # --- FIXED: Force re-evaluation on new structures ---
        case ('-', 0, a): return simplify(('neg', a))
        case ('-', a, ('neg', b)): return simplify(('+', a, b)) 
        
        case ('-', a, 0): return a
        case ('-', a, b) if a == b: return 0

        # Multiplication
        case ('*', 0, _) | ('*', _, 0): return 0
        case ('*', 1, a) | ('*', a, 1): return a
        
        # --- FIXED: Force re-evaluation on new structures ---
        case ('*', -1, a) | ('*', a, -1): return simplify(('neg', a))
        
        case ('*', a, ('/', 1, b)) if a == b: return 1
        case ('*', ('/', 1, b), a) if a == b: return 1
        case ('*', ('neg', a), ('/', 1, b)) if a == b: return -1
        case ('*', ('/', 1, b), ('neg', a)) if a == b: return -1
        case ('*', a, ('neg', ('/', 1, b))) if a == b: return -1
        case ('*', ('neg', ('/', 1, b)), a) if a == b: return -1

        # Power Derivative Reduction Rules
        case ('*', ('*', a, ('/', 1, f)), ('**', f2, b)) if f == f2 and isinstance(b, (int, float)):
            if b == 2: return ('*', a, f)
            return ('*', a, ('**', f, b - 1))
        case ('*', ('**', f2, b), ('*', a, ('/', 1, f))) if f == f2 and isinstance(b, (int, float)):
            if b == 2: return ('*', a, f)
            return ('*', a, ('**', f, b - 1))
        case ('*', ('/', 1, f), ('**', f2, b)) if f == f2 and isinstance(b, (int, float)):
            if b == 2: return f
            return ('**', f, b - 1)
        case ('*', ('**', f2, b), ('/', 1, f)) if f == f2 and isinstance(b, (int, float)):
            if b == 2: return f
            return ('**', f, b - 1)

        # Division
        case ('/', 0, _): return 0
        case ('/', a, 1): return a
        case ('/', a, b) if a == b: return 1

        # Exponentiation
        case ('**', _, 0): return 1
        case ('**', a, 1): return a
        case ('**', 0, _): return 0
        case ('**', 1, _): return 1

        # Logarithmic & Exponential Identities
        case ('ln', 1): return 0
        case ('ln', ('exp', a)): return a
        case ('exp', 0): return 1
        case ('exp', ('ln', a)): return a
        case ('exp', ('*', a, ('ln', f))): return ('**', f, a)
        case ('exp', ('*', ('ln', f), a)): return ('**', f, a)

        # Fallback
        case _: 
            return e_sim

## String Formatting
We have updated the `precedence` dictionary to appropriately rank the new unary `neg` operator.

In [ ]:
def toString(e):
    if isinstance(e, (int, float, str)):
        return str(e)
    if len(e) == 2:
        if e[0] == 'neg':
            # Only wrap in parens if inner expr is lower precedence than mult/div (+ or -)
            if precedenceOp(e[1]) <= 1:
                return '-(' + toString(e[1]) + ')'
            return '-' + toString(e[1])
        return e[0] + '(' + toString(e[1]) + ')'
    if e[0] == '+':
        return toString(e[1]) + ' + ' + toString(e[2])
    if e[0] == '-':
        lhs = toString(e[1])
        rhs = '(' + toString(e[2]) + ')' if precedenceOp(e[2]) == 1 else toString(e[2])
        return lhs + ' - ' + rhs
    if e[0] == '*':
        lhs = '(' + toString(e[1]) + ')' if precedenceOp(e[1]) == 1 else toString(e[1])
        rhs = '(' + toString(e[2]) + ')' if precedenceOp(e[2]) == 1 else toString(e[2])
        return lhs + '*' + rhs
    if e[0] == '/':
        lhs = '(' + toString(e[1]) + ')' if precedenceOp(e[1]) == 1 else toString(e[1])
        rhs = '(' + toString(e[2]) + ')' if precedenceOp(e[2]) <= 2 else toString(e[2])
        return lhs + '/' + rhs
    if e[0] == '**':
        lhs = '(' + toString(e[1]) + ')' if precedenceOp(e[1]) <= 4 else toString(e[1])
        rhs = '(' + toString(e[2]) + ')' if precedenceOp(e[2]) <= 3 else toString(e[2])
        return lhs + '**' + rhs

def precedenceOp(expr):
    if isinstance(expr, tuple):
        return precedence(expr[0])
    return 5

def precedence(op: str):
    Precedences = { '+': 1, '-': 1, '*': 2, '/': 2, 'neg': 3, '**': 4 }
    if op in Precedences:
        return Precedences[op]
    return 5

## Testing

In [ ]:
def test(s):
    t = parse(s)
    d = simplify(diff(t))
    print(f"d/dx {s} = {toString(d)}")

In [ ]:
test("ln(x ** x)")

In [ ]:
test("x ** -x")

In [ ]:
test("-x ** 2")

In [ ]:
test("1/(1+exp(-x))")

In [ ]:
test("x*x")

In [ ]:
test("x/x")